# 06e · Leakage fixes: physician-network feature and all-provider fills

Two fixes to `src/features.py`, measured on the training partition only.

- **(a) `AvgPhysicianProviderSpread`** counted, for each physician, the providers they bill under across the *whole* claims table, so test providers' claims shaped training providers' values. It is now computed against a reference set of providers (`reference_providers`): reference providers count themselves once, and any other provider is scored as if it alone had joined the reference set.
- **(b) `TopDiagnosisCodeShare`, `TopProcedureCodeShare`, `BenfordMAD`** were filled with a mean over all 5,410 providers (train + test). `build_provider_features()` now leaves them NaN for the model pipeline to impute from training rows. 06b/06c already reset these cells and impute per fold, so (b) changes no model score; it removes the trap for anyone using the table directly.

**Protocol.** Training partition and shared folds only (`artifacts/shared/cv_folds.csv`). In each fold the physician feature is recomputed with that fold's training providers as the reference, so neither validation-fold nor test providers influence it. **Test labels are never read**: the held-out set has been used once, so any new test comparison would be test-informed. The frozen model, threshold and evaluation are untouched.

| Section | What it does |
|---|---|
| 1 | Build features; check the baseline equals the frozen training inputs |
| 2 | How much the physician feature changes |
| 3 | Check the baseline reproduces the frozen out-of-fold predictions |
| 4 | Cross-validated effect of fix (a): frozen configurations, then a LightGBM re-search |
| 5 | Save leak-free train/test tables for any retraining |

In [1]:
import io
import json
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
from imblearn.pipeline import Pipeline as ImbPipeline
from lightgbm import LGBMClassifier
from scipy.stats import loguniform, randint, spearmanr, uniform
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, precision_recall_curve, roc_auc_score
from sklearn.model_selection import ParameterSampler
from sklearn.preprocessing import StandardScaler

sys.path.append("..")
from src.cleaning import clean_claim_level_dataset
from src.data_loader import build_claim_level_dataset
from src.features import build_physician_network_features, build_provider_features
from src.fold_vif import FoldVIFSelector

pd.set_option("display.float_format", "{:.4f}".format)
ROOT = Path("..")
SEED, N_FOLDS, N_JOBS, N_ITER, N_BOOT = 4012, 5, 4, 20, 2000
ID, TARGET, PHYS = "Provider", "PotentialFraud", "AvgPhysicianProviderSpread"
RESTORE_NA_COLS = ["TopDiagnosisCodeShare", "TopProcedureCodeShare", "BenfordMAD"]
LGBM_RUN = ROOT / "artifacts/boosting/20261011_232303_full"
LR_RUN = ROOT / "artifacts/logistic_random_forest/20261115_010242_full"
OUT_DIR = ROOT / "artifacts/leakage_check" / time.strftime("%Y%m%d_%H%M%S")
OUT_DIR.mkdir(parents=True, exist_ok=False)

lgbm_meta = json.loads((LGBM_RUN / "selected_metadata.json").read_text())
lr_meta = json.loads((LR_RUN / "selected_metadata.json").read_text())
features = lgbm_meta["feature_order"]
assert features == lr_meta["feature_order"]

folds_df = pd.read_csv(ROOT / "artifacts/shared/cv_folds.csv")
train_ids = folds_df[ID].tolist()   # frozen training providers, in the frozen row order
folds = folds_df["fold"].to_numpy()


def csv_round_trip(frame):
    """Pass values through the CSV format the frozen model's inputs were saved in (16 decimal places)."""
    buf = io.StringIO()
    frame.to_csv(buf, index=False)
    buf.seek(0)
    return pd.read_csv(buf)


print(f"{len(train_ids)} training providers, {N_FOLDS} shared folds; outputs -> {OUT_DIR}")

4328 training providers, 5 shared folds; outputs -> ../artifacts/leakage_check/20261115_133326


## 1. Build features and check the baseline

`build_provider_features(claims)` with no reference keeps the original all-provider physician definition, so on the training providers it must equal the frozen training file once 06b's NaN restoration is applied. Values pass through the same CSV format the frozen inputs were saved in, so the comparison is exact.

In [2]:
claims = clean_claim_level_dataset(build_claim_level_dataset())
all_feats = build_provider_features(claims)
test_ids = pd.read_csv(lgbm_meta["test_path"], usecols=[ID])[ID].tolist()   # provider IDs only, no labels
assert set(test_ids) == set(all_feats[ID]) - set(train_ids)

base = csv_round_trip(all_feats.set_index(ID).loc[train_ids].reset_index()[[ID] + features + [TARGET]])
frozen_train = pd.read_csv(lgbm_meta["train_path"])
for col in RESTORE_NA_COLS:
    frozen_train.loc[frozen_train[f"{col}Missing"] == 1, col] = np.nan
assert frozen_train[ID].tolist() == train_ids, "Training file order differs from the shared folds"
assert np.array_equal(base[features].to_numpy(float), frozen_train[features].to_numpy(float), equal_nan=True)
assert (base[TARGET].to_numpy() == folds_df[TARGET].to_numpy()).all()

print(f"{len(all_feats)} providers: {len(train_ids)} train, {len(test_ids)} test")
print("Baseline training features equal the frozen training inputs exactly (after NaN restoration).")
print("Missing values left for the pipeline:", base[RESTORE_NA_COLS].isna().sum().to_dict(),
      "| physician feature missing:", int(base[PHYS].isna().sum()))

5410 providers: 4328 train, 1082 test
Baseline training features equal the frozen training inputs exactly (after NaN restoration).
Missing values left for the pipeline: {'TopDiagnosisCodeShare': 4, 'TopProcedureCodeShare': 2712, 'BenfordMAD': 1674} | physician feature missing: 0


## 2. How much the physician feature changes

Original (all 5,410 providers as the reference) versus the leak-free version (the 4,328 training providers as the reference). Test providers' values are computed from their own claims only; no labels are involved.

In [3]:
full_ref = all_feats.set_index(ID)[PHYS]
train_ref = build_physician_network_features(claims, reference_providers=train_ids).set_index(ID)[PHYS]


def change_summary(ids, label):
    before, after = full_ref.reindex(ids), train_ref.reindex(ids)
    d = (after - before).abs()
    return {"providers": label, "n": len(ids), "share changed": float((d > 1e-12).mean()),
            "mean |change|": float(d.mean()), "max |change|": float(d.max()),
            "mean before": float(before.mean()), "mean after": float(after.mean()),
            "Spearman before vs after": float(spearmanr(before, after, nan_policy="omit").correlation)}


change = pd.DataFrame([change_summary(train_ids, "train"), change_summary(test_ids, "test")])
change.to_csv(OUT_DIR / "physician_feature_change.csv", index=False)

# Per-fold leak-free version for CV: each fold's training providers are the reference
fold_phys = {}
for k in range(N_FOLDS):
    reference = [p for p, f in zip(train_ids, folds) if f != k]
    spread = build_physician_network_features(claims, reference_providers=reference).set_index(ID)[PHYS]
    fold_phys[k] = csv_round_trip(spread.reindex(train_ids).to_frame())[PHYS].to_numpy()
change

,providers,n,share changed,mean |change|,max |change|,mean before,mean after,Spearman before vs after
0,train,4328,0.2960,0.0688,2.0000,1.3521,1.2833,0.8974
1,test,1082,0.2994,0.0596,1.0000,1.3677,1.3081,0.8963


## 3. Reproduce the frozen out-of-fold predictions

The pipelines below copy 06b (median imputation → LightGBM) and 06c (median imputation → scaling → fold-local VIF < 10 → L1 logistic regression), with each run's saved configuration. On the baseline features they should reproduce each run's saved OOF probabilities. If they do, any change in Section 4 comes from fix (a) alone.

In [4]:
def lgbm_pipe(params):
    return ImbPipeline([
        ("impute", SimpleImputer(strategy="median", keep_empty_features=True).set_output(transform="pandas")),
        ("model", LGBMClassifier(objective="binary", random_state=SEED, n_jobs=N_JOBS, verbose=-1, **params))])


def lr_pipe(params, vif):
    return ImbPipeline([
        ("impute", SimpleImputer(strategy="median", keep_empty_features=True).set_output(transform="pandas")),
        ("scale", StandardScaler().set_output(transform="pandas")),
        ("select", FoldVIFSelector(threshold=vif)),
        ("model", LogisticRegression(solver="liblinear", max_iter=5000, random_state=SEED, **params))])


X_base, y = base[features], base[TARGET].to_numpy()


def cross_validate(make, variant):
    oof, aps = np.full(len(y), np.nan), []
    for k in range(N_FOLDS):
        X = X_base.copy()
        if variant == "fixed":
            X[PHYS] = fold_phys[k]
        tr, va = np.where(folds != k)[0], np.where(folds == k)[0]
        est = make().fit(X.iloc[tr], y[tr])
        oof[va] = est.predict_proba(X.iloc[va])[:, 1]
        aps.append(average_precision_score(y[va], oof[va]))
    return oof, np.array(aps)


saved_lgbm = pd.read_csv(LGBM_RUN / "oof_predictions.csv").query("model == 'lgbm' and strategy == 'original'")
saved_lr = pd.read_csv(LR_RUN / "oof_predictions.csv").query("model == 'lr_vif10' and strategy == 'original'")
runs = {
    "LightGBM (frozen config)": (lambda: lgbm_pipe(lgbm_meta["params"]), saved_lgbm),
    "LR, fold VIF<10 (06c config)": (lambda: lr_pipe(lr_meta["params"], lr_meta["vif_threshold"]), saved_lr),
}
results = {}
for name, (make, saved) in runs.items():
    assert saved[ID].tolist() == train_ids
    oof_b, ap_b = cross_validate(make, "baseline")
    gap = np.max(np.abs(oof_b - saved["oof_prob"].to_numpy()))
    print(f"{name}: max |OOF - saved OOF| = {gap:.2e}; mean fold AP {ap_b.mean():.5f}")
    results[name] = {"baseline": (oof_b, ap_b)}

LightGBM (frozen config): max |OOF - saved OOF| = 1.11e-16; mean fold AP 0.67396


LR, fold VIF<10 (06c config): max |OOF - saved OOF| = 5.55e-16; mean fold AP 0.66473


## 4. Cross-validated effect of fix (a)

Same folds, same configuration, only the physician feature differs. Fold differences are paired. The bootstrap resamples providers within each class (2,000 draws, seed 4012) and holds the OOF predictions fixed, like the 06b/06c comparison; it does not include retraining uncertainty. "Precision at ≥80% recall" applies the project's threshold rule to each variant's OOF scores.

In [5]:
def precision_at_recall(y_true, p, target=0.80):
    prec, rec, thr = precision_recall_curve(y_true, p)
    ok = rec[:-1] >= target
    i = int(np.argmax(np.where(ok, prec[:-1], -1)))
    return float(prec[i]), float((p >= thr[i]).mean())


def bootstrap_ap_diff(y_true, p_new, p_old, n_boot=N_BOOT, seed=SEED):
    rng = np.random.default_rng(seed)
    pos, neg = np.where(y_true == 1)[0], np.where(y_true == 0)[0]
    diffs = []
    for _ in range(n_boot):
        idx = np.concatenate([rng.choice(pos, len(pos)), rng.choice(neg, len(neg))])
        diffs.append(average_precision_score(y_true[idx], p_new[idx]) - average_precision_score(y_true[idx], p_old[idx]))
    return np.quantile(diffs, [0.025, 0.975])


rows, fold_rows = [], []
for name, (make, _) in runs.items():
    oof_b, ap_b = results[name]["baseline"]
    oof_f, ap_f = cross_validate(make, "fixed")
    results[name]["fixed"] = (oof_f, ap_f)
    diff = ap_f - ap_b
    lo, hi = bootstrap_ap_diff(y, oof_f, oof_b)
    prec_b, share_b = precision_at_recall(y, oof_b)
    prec_f, share_f = precision_at_recall(y, oof_f)
    rows.append({"model": name,
                 "mean fold AP, baseline": ap_b.mean(), "mean fold AP, fixed": ap_f.mean(),
                 "mean fold diff": diff.mean(), "folds worse": int((diff < 0).sum()),
                 "pooled AP, baseline": average_precision_score(y, oof_b),
                 "pooled AP, fixed": average_precision_score(y, oof_f),
                 "pooled AP diff 95% CI": f"{lo:+.4f} to {hi:+.4f}",
                 "ROC AUC, baseline": roc_auc_score(y, oof_b), "ROC AUC, fixed": roc_auc_score(y, oof_f),
                 "precision at ≥80% recall, baseline": prec_b, "precision at ≥80% recall, fixed": prec_f,
                 "share reviewed, baseline": share_b, "share reviewed, fixed": share_f})
    fold_rows += [{"model": name, "fold": k, "baseline AP": ap_b[k], "fixed AP": ap_f[k], "diff": diff[k]}
                  for k in range(N_FOLDS)]

effect = pd.DataFrame(rows)
effect_folds = pd.DataFrame(fold_rows)
effect.to_csv(OUT_DIR / "effect_frozen_configs.csv", index=False)
effect_folds.to_csv(OUT_DIR / "effect_fold_metrics.csv", index=False)
display(effect_folds.pivot(index="fold", columns="model", values="diff").rename_axis(columns="fixed − baseline fold AP"))
effect.set_index("model").T

fixed − baseline fold AP,"LR, fold VIF<10 (06c config)",LightGBM (frozen config)
fold,,
0,0.0007,-0.0091
1,0.0000,0.0024
2,0.0000,-0.0103
3,0.0005,-0.0072
4,-0.0000,-0.0097


model,LightGBM (frozen config),"LR, fold VIF<10 (06c config)"
"mean fold AP, baseline",0.6740,0.6647
"mean fold AP, fixed",0.6672,0.6649
mean fold diff,-0.0068,0.0002
folds worse,4,1
"pooled AP, baseline",0.6720,0.6624
"pooled AP, fixed",0.6689,0.6625
pooled AP diff 95% CI,-0.0116 to +0.0060,-0.0006 to +0.0007
"ROC AUC, baseline",0.9353,0.9310
"ROC AUC, fixed",0.9340,0.9311
"precision at ≥80% recall, baseline",0.4622,0.4649


### LightGBM re-search

The frozen configuration was chosen on the baseline features, which could favour the baseline. This reruns 06b's 20 LightGBM configurations (same search space and sampler seed, original class balance) on both variants and compares the best of each.

In [6]:
LGBM_SPACE = {
    "n_estimators": randint(200, 800),
    "learning_rate": loguniform(0.01, 0.2),
    "num_leaves": randint(8, 64),
    "min_child_samples": randint(10, 60),
    "subsample": uniform(0.6, 0.4),
    "subsample_freq": [1],
    "colsample_bytree": uniform(0.5, 0.5),
    "reg_lambda": loguniform(1e-2, 10),
}
configs = [{k: (v.item() if hasattr(v, "item") else v) for k, v in c.items()}
           for c in ParameterSampler(LGBM_SPACE, n_iter=N_ITER, random_state=SEED)]
assert configs[lgbm_meta["config_id"]] == lgbm_meta["params"], "Sampler does not reproduce 06b's configurations"

search = []
t0 = time.time()
for i, params in enumerate(configs):
    for variant in ("baseline", "fixed"):
        _, aps = cross_validate(lambda: lgbm_pipe(params), variant)
        search.append({"config_id": i, "variant": variant, "ap_mean": aps.mean(), "ap_std": aps.std(ddof=1)})
search = pd.DataFrame(search)
search.to_csv(OUT_DIR / "lgbm_search.csv", index=False)
print(f"{len(configs) * 2 * N_FOLDS} fits in {time.time() - t0:.0f}s")

by_config = search.pivot(index="config_id", columns="variant", values="ap_mean")
by_config["diff"] = by_config["fixed"] - by_config["baseline"]
best = search.loc[search.groupby("variant")["ap_mean"].idxmax()].set_index("variant")
print(f"Configurations where the fixed feature scores higher: {(by_config['diff'] > 0).sum()} of {len(by_config)}; "
      f"mean diff {by_config['diff'].mean():+.5f}")
best

200 fits in 238s
Configurations where the fixed feature scores higher: 5 of 20; mean diff -0.00314


,config_id,ap_mean,ap_std
variant,,,
baseline,13,0.6740,0.0258
fixed,0,0.6723,0.0145


## 5. Save leak-free tables for any retraining

Built with the library defaults: no all-provider fill, and the physician feature computed with the training providers as the reference. The column layout matches `provider_features_{train,test}.csv`, so 06b/06c can use these by changing `TRAIN_PATH`/`TEST_PATH` (their NaN restoration is then a no-op). The canonical files stay unchanged because the frozen model and dashboard depend on them.

In [7]:
leakfree = build_provider_features(claims, reference_providers=train_ids).set_index(ID)
assert np.allclose(leakfree[PHYS].reindex(train_ids + test_ids), train_ref.reindex(train_ids + test_ids), equal_nan=True)
others = [c for c in features if c != PHYS]
assert np.array_equal(leakfree.loc[all_feats[ID], others].to_numpy(float),
                      all_feats.set_index(ID)[others].to_numpy(float), equal_nan=True)

cols = [ID] + features + [TARGET]
leakfree.loc[train_ids].reset_index()[cols].to_csv("../data/processed/provider_features_train_leakfree.csv", index=False)
leakfree.loc[test_ids].reset_index()[cols].to_csv("../data/processed/provider_features_test_leakfree.csv", index=False)

summary = {
    "physician_feature_change": change.to_dict(orient="records"),
    "frozen_config_effect": effect.to_dict(orient="records"),
    "lgbm_search_best": best.reset_index().to_dict(orient="records"),
    "lgbm_search_configs_improved": int((by_config["diff"] > 0).sum()),
    "n_boot": N_BOOT, "seed": SEED, "test_labels_read": False,
}
(OUT_DIR / "summary.json").write_text(json.dumps(summary, indent=2, default=float))
print("Saved data/processed/provider_features_{train,test}_leakfree.csv and", OUT_DIR / "summary.json")

Saved data/processed/provider_features_{train,test}_leakfree.csv and ../artifacts/leakage_check/20261115_133326/summary.json


## Findings

**The leak was small, and it slightly flattered LightGBM.**

- **Reproduction is exact.** On the baseline features, both pipelines reproduce their saved OOF probabilities (max difference 1.1 × 10⁻¹⁶ for LightGBM, 5.6 × 10⁻¹⁶ for LR), so every difference below comes from fix (a) alone.
- **The feature itself moves modestly.** Using the training providers as the reference changes `AvgPhysicianProviderSpread` for 30% of providers (mean |change| 0.07 on a mean of about 1.35; Spearman 0.90 between the two versions).
- **LightGBM, frozen configuration:** mean fold AP 0.6740 → 0.6672 (−0.0068, worse in 4 of 5 folds); pooled OOF AP 0.6720 → 0.6689, with a bootstrap interval for the difference of −0.0116 to +0.0060 that includes zero. At the project's ≥80%-recall rule, precision is essentially unchanged (0.462 → 0.462) for the same review share (16.2%).
- **LightGBM, re-searched:** the best of 06b's 20 configurations scores 0.6723 on the fixed feature versus 0.6740 on the original (−0.0017); 5 of 20 configurations improve.
- **Logistic regression (fold VIF < 10):** no change (+0.0002 mean fold AP, interval −0.0006 to +0.0007).

**How to read it.** This was covariate (transductive) leakage, not target leakage: no labels entered the feature, but other providers' claims, including test providers', did. The original version also simply counted physicians over more providers, so part of its edge is extra information rather than test-set contamination. Either way, the leak-free figure is the honest one for "a model trained only on training providers". The frozen model's validation AP is optimistic by about 0.002–0.007 on this evidence, which is well inside the held-out AP interval (0.680–0.827) and does not change the model choice. Its effect on the held-out result itself cannot be measured without reusing test labels, so it is not estimated here.

**What changes going forward.**
- Any retraining should use `provider_features_{train,test}_leakfree.csv`, or call `build_provider_features(df, reference_providers=<training providers>)`, and tune on that.
- The frozen model, its threshold, the held-out evaluation and the dashboard are unchanged. The canonical tables still reproduce the frozen fingerprints, which `run_pipeline.sh` checks.
- Fix (b) needs no measurement: the model notebooks already imputed these three columns inside each fold. It now holds by default for anyone who uses the library or the leak-free tables directly.